# Data Preparation

In [23]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

con = sqlite3.connect("../data/hockey.db")
q = lambda sql: pd.read_sql(sql, con)
sql_file = lambda name: q(open(f"../sql/{name}").read())

sns.set_theme(style="whitegrid")

## Data Integration

In this section we join tables so that each row carries information from more than one source. The rubric asks for two things, and we document both:

- **Matching entities between sources**: the key used to link two tables.
- **Conversion of formats**: turning raw fields into values a model can use.

### Coach profile

Query: [`sql/coach_profile.sql`](../sql/coach_profile.sql). One row per coach stint (`coachID`, `year`, `tmID`, `lgID`, `stint`), for **all leagues**, with the person's details from `Master`.

| Source | Target | Key | What we bring in |
|---|---|---|---|
| `Coaches` | `Master` | `coachID` | birth date, `playerID`, `firstNHL` |

- We use a **left join** with `Coaches` as the driving table, so every coaching stint is kept even if the person were missing from `Master`.
- The result has **1,741 rows**, the same as `Coaches`. Every coach finds a match, and no `coachID` appears twice in `Master`, so the join neither loses nor duplicates rows.
- We keep **all leagues** (NHL, WHA, NHA, PCHA, WCHL), not only the NHL, so that a coach's experience can later include seasons coached outside the NHL. `lgID` is kept so we can filter to the NHL when building targets, and to join with `Teams` later.

#### Age at season start (format conversion)

`Master` stores the birth date in three separate columns (`birthYear`, `birthMon`, `birthDay`). We convert them into one number: the coach's **age at the start of the season**.

- **Reference date:** 1 August of `year`. In this dataset `year` is the season's starting year (2005 = 2005-06), so 1 August is before the season begins, and the age is known at prediction time.
- **Birthday rule:** if the birthday falls after 1 August, the coach has not turned the new age yet, so we subtract one year.
- **Missing month or day:** we assume 1 August, the same as the reference date. The age then becomes simply `year − birthYear`, which is wrong by at most one year.
- **Missing year:** the age is left missing (`<NA>`). It is not imputed here; that is handled in the *Missing values* section.

In [24]:
import datetime

coaches = sql_file("coach_profile.sql")

def calculate_age(row):
    if pd.isna(row["birthYear"]):
        return pd.NA
    
    birth_year = int(row["birthYear"])
    birth_month = 8 if pd.isna(row["birthMon"]) else int(row["birthMon"])
    birth_day = 1 if pd.isna(row["birthDay"]) else int(row["birthDay"])
    
    birth_date = datetime.date(
        birth_year,
        birth_month,
        birth_day
    )

    reference_date = datetime.date(
        int(row["year"]),
        8,
        1
    )

    age = reference_date.year - birth_date.year

    if (birth_date.month, birth_date.day) > (
        reference_date.month,
        reference_date.day
    ):
        age -= 1

    return age

coaches["age"] = coaches.apply(calculate_age, axis=1)
coaches["age"] = coaches["age"].astype("Int64")

#### Former player (format conversion)

`Master` only tells us *whether* a coach has a player record. We turn that into two 0/1 features:

| Feature | Rule | Meaning |
|---|---|---|
| `former_player` | `playerID` is not null | played in any league in the data |
| `former_nhl_player` | `firstNHL` is not null | played in the NHL |

Here a null does **not** mean "unknown": it means the person never played. So these flags have no missing values.

After building the features we drop the raw columns (`birthYear`, `birthMon`, `birthDay`, `playerID`, `firstNHL`), which are not useful to a model as they are.

In [25]:
import numpy as np
coaches["former_player"] = np.where(
    coaches["playerID"].isna(), 0, 1
)
coaches["former_nhl_player"] = np.where(
    coaches["firstNHL"].isna(), 0, 1
)

In [26]:
df = coaches.drop(["birthYear", "birthMon", "birthDay", "playerID", "firstNHL"], axis = 1)

#### Checks on the coach profile

In [27]:
younger_coach = df["age"].min()
oldest_coach = df["age"].max()
print(f'The youngest coach as age: {younger_coach}. The Oldest coach as age: {oldest_coach}')

The youngest coach as age: 22. The Oldest coach as age: 74


In [28]:
no_age = df["age"].isnull().sum()
print(f'There is {no_age} missing values')

null_age = df[df["age"].isnull()]
grouped_no_age = null_age.groupby('lgID').size().reset_index(name="count").sort_values("count",ascending=False)

print(grouped_no_age)

There is 60 missing values
   lgID  count
1   NHL     25
0   NHA     19
3  WCHL     10
2  PCHA      3
4   WHA      3


In [29]:
total = df["former_player"].count()
former_player = (df["former_player"] == 1).sum()

percentage = former_player / total * 100

print(f'The percentage of coaches that were former players is {percentage:.2f}%')

total = df["former_nhl_player"].count()
former_nhl_player = (df["former_nhl_player"] == 1).sum()

percentage = former_nhl_player / total * 100

print(f'The percentage of coaches that were former players is {percentage:.2f}%')


The percentage of coaches that were former players is 65.54%
The percentage of coaches that were former players is 63.12%


**Results**

- **Age range: 22 to 74.** Both are real cases, not data errors. The youngest is a player-coach in the NHA in 1916, when teams often had a player also coaching. The oldest is Al Arbour, who came back for one game with the Islanders in 2007. The youngest coach in the NHL itself is 25 (1979). No age is impossible, so no value is removed (validity).
- **60 stints (3.4%) have no age**, from 27 coaches with no `birthYear`. They come mostly from the early leagues (NHA, PCHA, WCHL). The 25 NHL rows are all from before 1940, so this gap should almost disappear after the era selection. We re-check it there (completeness).
- **Missing day or month only:** 30 more stints have a birth year but no month or day; their age uses the approximation described above.
- **Former players:** 65.5% of coaching stints are by former players, and 63.1% by former NHL players. These percentages count **stints (rows)**, not distinct coaches, so coaches with long careers count more than once.

## Data Quality

## Era Selection

## Targets

## Feature engineering

## Missing values

## Outliers

## Redundancy

## Save the Final table